In [7]:
# ==========================================
# NB_SILVER: BRONZE -> SILVER
# CLEANSED + VALIDATED + QUARANTINE VERSION
# ==========================================
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

print("==================================================")
print("BẮT ĐẦU XỬ LÝ SILVER LAYER")
print("==================================================")

# ==========================================
# BƯỚC 1: ĐỌC DỮ LIỆU BRONZE (HỖ TRỢ SNAPSHOT LỊCH SỬ BẰNG DẤU *)
# ==========================================
try:
    print("[READ] Đang đọc dữ liệu Bronze...")
    # Quét toàn bộ các thư mục con snapshot động theo thời gian
    raw_sales_df = spark.read.parquet("Files/Bronze/sales/*")
    raw_exchange_df = spark.read.parquet("Files/Bronze/exchange_rate/*")

    print(f"[SALES RAW] {raw_sales_df.count()} records")
    print(f"[EXCHANGE RAW] {raw_exchange_df.count()} records")
except Exception as e:
    print(f"[ERROR] Không đọc được Bronze Layer: {str(e)}")
    raise

# ==========================================
# BƯỚC 2: DATA CLEANSING SALES
# ==========================================
print("[PROCESSING] Làm sạch bảng Sales...")

cleaned_sales_df = raw_sales_df \
    .withColumn("order_id", trim(col("order_id"))) \
    .withColumn(
    "order_date",
    coalesce(
        to_timestamp(col("order_date"), "yyyy-MM-dd'T'HH:mm:ss.SSSSSS"),
        to_timestamp(col("order_date"), "dd/MM/yyyy HH:mm")
    )
        )\
    .withColumn(
        "customer_age",
        when(lower(trim(col("customer_age"))) == "thirty-five", 35)
        .otherwise(col("customer_age").cast(IntegerType()))
    ) \
    .withColumn("location", coalesce(trim(col("location")), lit("Unknown"))) \
    .withColumn("device_type", coalesce(trim(col("device_type")), lit("Unknown"))) \
    .withColumn("payment_method", coalesce(trim(col("payment_method")), lit("Unknown"))) \
    .withColumn("currency", upper(trim(col("currency")))) \
    .withColumn(
        "discount_code",
        when(trim(col("discount_code")) == "", None).otherwise(trim(col("discount_code")))
    ) \
    .withColumn("shipping_cost", col("shipping_cost").cast(DoubleType())) \
    .withColumn("total_amount", col("total_amount").cast(DoubleType())) \
    .withColumn("order_status", coalesce(trim(col("order_status")), lit("Unknown"))) \
    .withColumn("loyalty_points", col("loyalty_points").cast(IntegerType())) \
    .withColumn("feedback_score", col("feedback_score").cast(IntegerType())) \
    .withColumn("ingestion_time", current_timestamp())

# ==========================================
# BƯỚC 3: BUSINESS VALIDATION SALES (PHÂN TÁCH DATA LỖI)
# ==========================================
print("[VALIDATION] Kiểm tra dữ liệu Sales...")

valid_sales_condition = (
    col("order_id").isNotNull() & 
    col("order_date").isNotNull() & 
    (col("customer_age").isNull() | (col("customer_age") >= 0)) & 
    (col("feedback_score").isNull() | col("feedback_score").between(1, 5))
)

sales_quarantine_df = cleaned_sales_df.filter(~valid_sales_condition) \
    .withColumn("quarantine_reason", 
                when(col("order_id").isNull(), "Missing Order ID")
                .when(col("order_date").isNull(), "Invalid Date Format")
                .when(col("customer_age") < 0, "Negative Customer Age")
                .otherwise("Invalid Feedback Score"))

final_sales_df = cleaned_sales_df.filter(valid_sales_condition)

# ==========================================
# BƯỚC 4: REMOVE DUPLICATES (BATCH LEVEL)
# ==========================================
before_dedup = final_sales_df.count()
final_sales_df = final_sales_df.dropDuplicates(["order_id"])
after_dedup = final_sales_df.count()
duplicate_removed = before_dedup - after_dedup
print(f"[DEDUP] Đã loại bỏ {duplicate_removed} bản ghi trùng trong Batch")

# ==========================================
# BƯỚC 5: DATA CLEANSING EXCHANGE RATE
# ==========================================
print("[PROCESSING] Làm sạch bảng Exchange Rate...")

cleaned_exchange_df = raw_exchange_df \
    .withColumn("year", col("year").cast(IntegerType())) \
    .withColumn("month", col("month").cast(IntegerType())) \
    .withColumn("from_currency", upper(trim(col("from_currency")))) \
    .withColumn("to_currency", upper(trim(col("to_currency")))) \
    .withColumn("exchange_rate", col("exchange_rate").cast(DoubleType())) \
    .withColumn("ingestion_time", current_timestamp())

valid_exchange_condition = (
    col("year").isNotNull() & 
    col("month").isNotNull() & 
    col("exchange_rate").isNotNull() & 
    (col("exchange_rate") > 0)
)

exchange_quarantine_df = cleaned_exchange_df.filter(~valid_exchange_condition)
final_exchange_df = cleaned_exchange_df.filter(valid_exchange_condition)

# ==========================================
# BƯỚC 6: GHI SILVER TABLE (IDEMPOTENT OVERWRITE)
# ==========================================
print("[WRITE] Ghi Silver Tables...")

final_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_sales")

final_exchange_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_exchange_rate")

# ==========================================
# BƯỚC 7: GHI QUARANTINE TABLE (LƯU TÍCH LŨY LỊCH SỬ LỖI)
# ==========================================
print("[WRITE] Tích lũy lịch sử bảng cách ly dữ liệu lỗi...")

sales_quarantine_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver_sales_quarantine")

exchange_quarantine_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver_exchange_quarantine")

# ==========================================
# BƯỚC 8: THỐNG KÊ KẾT QUẢ
# ==========================================
sales_count = final_sales_df.count()
exchange_count = final_exchange_df.count()
sales_quarantine_count = sales_quarantine_df.count()
exchange_quarantine_count = exchange_quarantine_df.count()

print("===================================")
print("THỐNG KÊ SILVER")
print("===================================")
print(f"Silver Sales (Sạch): {sales_count}")
print(f"Silver Exchange (Sạch): {exchange_count}")
print(f"Sales Quarantine (Mới phát hiện): {sales_quarantine_count}")
print(f"Exchange Quarantine (Mới phát hiện): {exchange_quarantine_count}")

# ==========================================
# BƯỚC 9: AUDIT LOGGING
# ==========================================
try:
    notebook_name = "NB_SILVER"
    status = "SUCCESS"
    execution_time = datetime.now()

    log_data = [(
        notebook_name,
        status,
        execution_time,
        int(sales_count),
        int(sales_quarantine_count)
    )]

    log_schema = [
        "notebook_name",
        "status",
        "execution_at",
        "processed_records",
        "quarantine_records"
    ]

    pipeline_log_df = spark.createDataFrame(log_data, schema=log_schema)
    pipeline_log_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("sys_pipeline_audit_logs")
except Exception as e:
    print(f"[WARN] Audit Log Error: {str(e)}")

print("==================================================")
print("HOÀN THÀNH SILVER LAYER SUCCESSFUL")
print("==================================================")

StatementMeta(, 81a3a6f8-49be-41c1-b45a-61bbe97cbd12, 9, Finished, Available, Finished, False)

BẮT ĐẦU XỬ LÝ SILVER LAYER
[READ] Đang đọc dữ liệu Bronze...
[SALES RAW] 10500 records
[EXCHANGE RAW] 48 records
[PROCESSING] Làm sạch bảng Sales...
[VALIDATION] Kiểm tra dữ liệu Sales...
[DEDUP] Đã loại bỏ 4704 bản ghi trùng trong Batch
[PROCESSING] Làm sạch bảng Exchange Rate...
[WRITE] Ghi Silver Tables...
[WRITE] Tích lũy lịch sử bảng cách ly dữ liệu lỗi...
THỐNG KÊ SILVER
Silver Sales (Sạch): 4302
Silver Exchange (Sạch): 48
Sales Quarantine (Mới phát hiện): 1494
Exchange Quarantine (Mới phát hiện): 0
HOÀN THÀNH SILVER LAYER SUCCESSFUL
